# KIT719 Project 1: Information Retrieval Application
## Complete standalone Jupyter Notebook

**Group members:** `[Name 1]`, `[Name 2/3]`  
**Student IDs:** `[ID 1]`, `[ID 2/3]`  
**Dataset:** Reuters Corpus (NLTK)

This notebook contains the complete implementation directly in notebook cells. It does not depend on a separate project Python module.

The application implements:

- dataset selection and document collection;
- text preprocessing;
- document indexing;
- TF-IDF and cosine-similarity ranking;
- BM25 ranking;
- WordNet-based query expansion;
- retrieval evaluation using Precision@K, Recall@K and MAP@K;
- optional named-entity and sentiment analysis; and
- a user-facing search interface.


## 1. Environment setup

Run the following cell once on a new environment. Package-installation commands are commented out so that normal notebook execution does not reinstall packages every time.


In [ ]:
# Install packages only when required.
# %pip install nltk numpy pandas matplotlib scikit-learn spacy

import nltk

NLTK_PACKAGES = [
    "reuters",
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "vader_lexicon",
]

for package in NLTK_PACKAGES:
    nltk.download(package, quiet=True)

print("NLTK resources are ready.")

# Install the spaCy English model once if it is not available:
# !python -m spacy download en_core_web_sm


## 2. Imports and application configuration

This section imports the libraries used throughout the information retrieval application and defines shared configuration values.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

import math
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from nltk import FreqDist, pos_tag
from nltk.corpus import reuters, stopwords, wordnet
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


DEFAULT_CATEGORIES = [
    "acq",
    "crude",
    "trade",
    "money-fx",
    "interest",
    "ship",
]

NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}
ENTITY_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Results directory:", RESULTS_DIR.resolve())


## 3. Ranked-result data structure

`RetrievalResult` stores each ranked document in a consistent format for display and evaluation.


In [ ]:
@dataclass(frozen=True)
class RetrievalResult:
    """One ranked document returned by the retrieval application."""

    rank: int
    doc_index: int
    doc_id: str
    categories: tuple[str, ...]
    score: float
    preview: str

## 4. Text preprocessing module

The preprocessing pipeline tokenises text, converts it to lowercase, removes English stop words, and applies Porter or Snowball stemming. The same pipeline is applied to documents and user queries.


In [ ]:
class TextPreprocessingModule:
    """Text preprocessing module for documents and user queries.

    The pipeline tokenises text, applies lowercase normalisation, removes
    English stop words, and performs Porter or Snowball stemming.
    """

    def __init__(self, stemmer_name: str = "porter") -> None:
        self.tokenizer = RegexpTokenizer(r"[A-Za-z]+")
        self.stop_words = set(stopwords.words("english"))

        stemmer_name = stemmer_name.lower().strip()
        if stemmer_name == "porter":
            self.stemmer = PorterStemmer()
        elif stemmer_name == "snowball":
            self.stemmer = SnowballStemmer("english")
        else:
            raise ValueError("stemmer_name must be 'porter' or 'snowball'")

        self.stemmer_name = stemmer_name

    def tokenise_and_filter(self, text: str) -> list[str]:
        """Tokenise, lowercase, and remove stop words without stemming."""
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        words: list[str] = []
        for token in self.tokenizer.tokenize(text):
            word = token.lower()
            if word not in self.stop_words:
                words.append(word)
        return words

    def preprocess_text(self, text: str) -> list[str]:
        """Return normalised and stemmed tokens."""
        return [self.stemmer.stem(word) for word in self.tokenise_and_filter(text)]

    def preprocess_query_terms(self, terms: Iterable[str]) -> list[str]:
        """Normalise and stem a sequence of words or short phrases."""
        output: list[str] = []
        for term in terms:
            output.extend(self.preprocess_text(term))
        return output

## 5. BM25 document index

The BM25 index stores term frequencies, document frequencies and document lengths. It calculates relevance scores using term-frequency saturation and document-length normalisation.


In [ ]:
class BM25Index:
    """Transparent Okapi BM25 index used by the ranking module."""

    def __init__(
        self,
        tokenised_documents: Sequence[Sequence[str]],
        k1: float = 1.5,
        b: float = 0.75,
    ) -> None:
        if not tokenised_documents:
            raise ValueError("tokenised_documents must not be empty")
        if k1 <= 0:
            raise ValueError("k1 must be positive")
        if not 0 <= b <= 1:
            raise ValueError("b must be between 0 and 1")

        self.documents = [list(tokens) for tokens in tokenised_documents]
        self.k1 = k1
        self.b = b
        self.total_documents = len(self.documents)
        self.document_lengths = [len(tokens) for tokens in self.documents]
        self.average_document_length = (
            sum(self.document_lengths) / self.total_documents
        )
        self.term_frequencies = [Counter(tokens) for tokens in self.documents]

        self.document_frequency: Counter[str] = Counter()
        for tokens in self.documents:
            self.document_frequency.update(set(tokens))

    def score(self, query_tokens: Sequence[str], doc_index: int) -> float:
        """Calculate the BM25 score of one document for a query."""
        if not 0 <= doc_index < self.total_documents:
            raise IndexError("doc_index is out of range")

        term_frequency = self.term_frequencies[doc_index]
        document_length = self.document_lengths[doc_index]
        score = 0.0

        # Counter avoids over-weighting duplicate query terms accidentally.
        query_counts = Counter(query_tokens)

        for term, query_count in query_counts.items():
            frequency = term_frequency.get(term, 0)
            if frequency == 0:
                continue

            df = self.document_frequency.get(term, 0)
            idf = math.log(
                1
                + (self.total_documents - df + 0.5)
                / (df + 0.5)
            )

            length_ratio = document_length / self.average_document_length
            denominator = frequency + self.k1 * (
                1 - self.b + self.b * length_ratio
            )
            tf_component = frequency * (self.k1 + 1) / denominator
            score += query_count * idf * tf_component

        return score

## 6. Information retrieval application

The main application constructs the inverted index, TF-IDF document matrix and BM25 index. It provides retrieval methods for TF-IDF, BM25 and BM25 with WordNet query expansion.


In [ ]:
class InformationRetrievalApplication:
    """Information retrieval application for the Reuters document collection."""

    def __init__(
        self,
        documents: list[dict[str, Any]],
        stemmer_name: str = "porter",
        k1: float = 1.5,
        b: float = 0.75,
    ) -> None:
        if not documents:
            raise ValueError("documents must not be empty")

        self.documents = documents
        self.preprocessor = TextPreprocessingModule(stemmer_name=stemmer_name)
        self.document_tokens = [
            self.preprocessor.preprocess_text(document["text"])
            for document in documents
        ]

        for document, tokens in zip(self.documents, self.document_tokens):
            document["tokens"] = tokens

        self.inverted_index = self._build_inverted_index(self.document_tokens)

        # Build the assignment TF-IDF document representation.
        self.vectorizer = TfidfVectorizer(
            analyzer=self.preprocessor.preprocess_text,
            lowercase=False,
            token_pattern=None,
            norm="l2",
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(
            [document["text"] for document in self.documents]
        )

        self.bm25 = BM25Index(self.document_tokens, k1=k1, b=b)

    @staticmethod
    def _build_inverted_index(
        tokenised_documents: Sequence[Sequence[str]],
    ) -> dict[str, list[int]]:
        """Map each term to the document indices that contain it."""
        index: defaultdict[str, list[int]] = defaultdict(list)
        for doc_index, tokens in enumerate(tokenised_documents):
            for term in set(tokens):
                index[term].append(doc_index)
        return dict(index)

    def analyse_term_frequency(self, top_n: int = 20) -> list[tuple[str, int]]:
        """Return the most frequent preprocessed terms for corpus analysis."""
        if top_n <= 0:
            raise ValueError("top_n must be positive")
        all_tokens = [
            token
            for document_tokens in self.document_tokens
            for token in document_tokens
        ]
        return FreqDist(all_tokens).most_common(top_n)

    def retrieve_candidate_documents(self, query_tokens: Sequence[str]) -> set[int]:
        """Use the inverted index to retrieve matching candidate documents."""
        candidates: set[int] = set()
        for token in query_tokens:
            candidates.update(self.inverted_index.get(token, []))
        return candidates

    def _format_results(
        self,
        ranked_pairs: Sequence[tuple[int, float]],
        top_k: int,
    ) -> list[RetrievalResult]:
        output: list[RetrievalResult] = []
        for rank, (doc_index, score) in enumerate(ranked_pairs[:top_k], start=1):
            document = self.documents[doc_index]
            preview = " ".join(document["text"].split())[:180]
            output.append(
                RetrievalResult(
                    rank=rank,
                    doc_index=doc_index,
                    doc_id=document["id"],
                    categories=tuple(document["categories"]),
                    score=float(score),
                    preview=preview,
                )
            )
        return output

    def rank_documents_tfidf(self, query_text: str, top_k: int = 10) -> list[RetrievalResult]:
        """Rank documents using TF-IDF vectors and cosine similarity."""
        if top_k <= 0:
            raise ValueError("top_k must be positive")

        query_tokens = self.preprocessor.preprocess_text(query_text)
        if not query_tokens:
            return []

        candidates = self.retrieve_candidate_documents(query_tokens)
        if not candidates:
            return []

        query_vector = self.vectorizer.transform([query_text])
        candidate_list = sorted(candidates)
        candidate_matrix = self.tfidf_matrix[candidate_list]
        scores = cosine_similarity(query_vector, candidate_matrix).ravel()

        ranked_pairs = sorted(
            zip(candidate_list, scores),
            key=lambda item: item[1],
            reverse=True,
        )
        return self._format_results(ranked_pairs, top_k)

    def _rank_documents_bm25_from_tokens(
        self,
        query_tokens: Sequence[str],
        top_k: int = 10,
    ) -> list[RetrievalResult]:
        """Rank documents using preprocessed query tokens and BM25."""
        if top_k <= 0:
            raise ValueError("top_k must be positive")
        if not query_tokens:
            return []

        candidates = self.retrieve_candidate_documents(query_tokens)
        if not candidates:
            return []

        ranked_pairs = [
            (doc_index, self.bm25.score(query_tokens, doc_index))
            for doc_index in candidates
        ]
        ranked_pairs.sort(key=lambda item: item[1], reverse=True)
        return self._format_results(ranked_pairs, top_k)

    def rank_documents_bm25(self, query_text: str, top_k: int = 10) -> list[RetrievalResult]:
        """Rank documents using BM25."""
        query_tokens = self.preprocessor.preprocess_text(query_text)
        return self._rank_documents_bm25_from_tokens(query_tokens, top_k=top_k)

    def expand_query_with_wordnet(
        self,
        query_text: str,
        max_synonyms_per_noun: int = 2,
    ) -> tuple[list[str], list[str]]:
        """Expand noun terms using ordered WordNet senses.

        Returns:
            raw_expanded_terms: original clean words plus added synonyms.
            processed_tokens: tokens ready for indexing and ranking.
        """
        if max_synonyms_per_noun < 0:
            raise ValueError("max_synonyms_per_noun must not be negative")

        original_words = self.preprocessor.tokenise_and_filter(query_text)
        if not original_words:
            return [], []

        tagged_words = pos_tag(original_words)
        expanded_terms = list(original_words)

        for word, tag in tagged_words:
            if tag not in NOUN_TAGS:
                continue

            added = 0
            # Inspect WordNet senses in their supplied order and cap expansion
            # to reduce query drift.
            for synset in wordnet.synsets(word, pos=wordnet.NOUN):
                for lemma in synset.lemmas():
                    synonym = lemma.name().lower().replace("_", " ")
                    if " " in synonym or synonym == word:
                        continue
                    if synonym not in expanded_terms:
                        expanded_terms.append(synonym)
                        added += 1
                    if added >= max_synonyms_per_noun:
                        break
                if added >= max_synonyms_per_noun:
                    break

        processed_tokens = self.preprocessor.preprocess_query_terms(expanded_terms)
        # Preserve order while removing duplicate stemmed tokens.
        processed_tokens = list(dict.fromkeys(processed_tokens))
        return expanded_terms, processed_tokens

    def rank_documents_bm25_with_query_expansion(
        self,
        query_text: str,
        top_k: int = 10,
        max_synonyms_per_noun: int = 2,
    ) -> list[RetrievalResult]:
        """Apply noun-focused WordNet expansion, then rank using BM25."""
        _, expanded_tokens = self.expand_query_with_wordnet(
            query_text,
            max_synonyms_per_noun=max_synonyms_per_noun,
        )
        return self._rank_documents_bm25_from_tokens(expanded_tokens, top_k=top_k)

## 7. Dataset loading and corpus-summary functions

These functions construct the Reuters document collection and summarise category distribution.


In [ ]:
def load_reuters_document_collection(
    categories: Sequence[str] | None = None,
) -> list[dict[str, Any]]:
    """Load the assignment document collection from selected Reuters categories."""
    selected_categories = list(categories or DEFAULT_CATEGORIES)
    document_ids: list[str] = []
    seen: set[str] = set()

    for category in selected_categories:
        for file_id in reuters.fileids(category):
            if file_id not in seen:
                seen.add(file_id)
                document_ids.append(file_id)

    documents: list[dict[str, Any]] = []
    for file_id in document_ids:
        documents.append(
            {
                "id": file_id,
                "text": reuters.raw(file_id),
                "categories": reuters.categories(file_id),
            }
        )
    return documents


def summarise_document_categories(documents: Sequence[dict[str, Any]]) -> pd.DataFrame:
    """Summarise the category distribution of the assignment dataset."""
    counts: Counter[str] = Counter()
    for document in documents:
        counts.update(document["categories"])

    rows = [
        {"category": category, "document_count": count}
        for category, count in sorted(counts.items())
    ]
    return pd.DataFrame(rows)

## 8. Retrieval evaluation functions

Reuters category labels are used as reproducible proxy relevance judgements. The system evaluates Precision@K, Recall@K, Average Precision@K and MAP@K.


In [ ]:
def precision_at_k(
    retrieved: Sequence[int],
    relevant: set[int],
    k: int,
) -> float:
    """Proportion of the first k retrieved documents that are relevant."""
    if k <= 0:
        raise ValueError("k must be positive")
    top_k = list(retrieved[:k])
    if not top_k:
        return 0.0
    hits = sum(doc_index in relevant for doc_index in top_k)
    # Precision@k convention uses k as denominator; missing ranks count as misses.
    return hits / k


def recall_at_k(
    retrieved: Sequence[int],
    relevant: set[int],
    k: int,
) -> float:
    """Proportion of all relevant documents found in the first k results."""
    if k <= 0:
        raise ValueError("k must be positive")
    if not relevant:
        return 0.0
    hits = sum(doc_index in relevant for doc_index in retrieved[:k])
    return hits / len(relevant)


def average_precision_at_k(
    retrieved: Sequence[int],
    relevant: set[int],
    k: int,
) -> float:
    """Calculate standard average precision truncated at k."""
    if k <= 0:
        raise ValueError("k must be positive")
    if not relevant:
        return 0.0

    hit_count = 0
    precision_sum = 0.0
    for rank, doc_index in enumerate(retrieved[:k], start=1):
        if doc_index in relevant:
            hit_count += 1
            precision_sum += hit_count / rank

    denominator = min(len(relevant), k)
    return precision_sum / denominator if denominator else 0.0


def relevant_document_indices(
    documents: Sequence[dict[str, Any]],
    category: str,
) -> set[int]:
    """Use Reuters category tags as reproducible proxy relevance labels."""
    return {
        index
        for index, document in enumerate(documents)
        if category in document["categories"]
    }


def evaluate_retrieval_performance(
    application: InformationRetrievalApplication,
    test_queries: dict[str, str],
    k: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate all implemented ranking methods using labelled test queries."""
    method_functions = {
        "TF-IDF + Cosine": application.rank_documents_tfidf,
        "BM25": application.rank_documents_bm25,
        "BM25 + WordNet Expansion": application.rank_documents_bm25_with_query_expansion,
    }

    detail_rows: list[dict[str, Any]] = []

    for category, query_text in test_queries.items():
        relevant = relevant_document_indices(application.documents, category)

        for method_name, search_function in method_functions.items():
            results = search_function(query_text, top_k=k)
            retrieved = [result.doc_index for result in results]

            detail_rows.append(
                {
                    "category": category,
                    "query": query_text,
                    "method": method_name,
                    f"precision@{k}": precision_at_k(retrieved, relevant, k),
                    f"recall@{k}": recall_at_k(retrieved, relevant, k),
                    f"AP@{k}": average_precision_at_k(retrieved, relevant, k),
                    "retrieved_count": len(retrieved),
                    "relevant_count": len(relevant),
                }
            )

    details = pd.DataFrame(detail_rows)
    summary = (
        details.groupby("method", as_index=False)[
            [f"precision@{k}", f"recall@{k}", f"AP@{k}"]
        ]
        .mean()
        .rename(columns={f"AP@{k}": f"MAP@{k}"})
        .sort_values(f"MAP@{k}", ascending=False)
        .reset_index(drop=True)
    )
    return details, summary

## 9. Optional result-enrichment functions

Named-entity recognition and sentiment analysis describe retrieved documents but do not change their relevance scores.


In [ ]:
def load_spacy_model(model_name: str = "en_core_web_sm") -> Any | None:
    """Load spaCy model; return None when the optional model is unavailable."""
    try:
        import spacy

        return spacy.load(model_name)
    except (ImportError, OSError):
        return None


def extract_document_entities(
    text: str,
    nlp: Any,
    allowed_labels: set[str] | None = None,
) -> list[tuple[str, str]]:
    """Extract selected named entities from a retrieved document."""
    labels = allowed_labels or ENTITY_LABELS
    document = nlp(text)
    return [
        (entity.text, entity.label_)
        for entity in document.ents
        if entity.label_ in labels
    ]


def initialise_sentiment_analyser() -> SentimentIntensityAnalyzer | None:
    """Create VADER analyser; return None until vader_lexicon is downloaded."""
    try:
        return SentimentIntensityAnalyzer()
    except LookupError:
        return None


def analyse_document_sentiment(
    text: str,
    analyser: SentimentIntensityAnalyzer,
) -> tuple[str, dict[str, float]]:
    """Return a VADER sentiment label and scores for result enrichment."""
    scores = analyser.polarity_scores(text)
    compound = scores["compound"]
    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"
    return label, scores

In [ ]:
def format_retrieval_results(results: Sequence[RetrievalResult]) -> pd.DataFrame:
    """Convert ranked retrieval results into a display-friendly DataFrame."""
    return pd.DataFrame(
        [
            {
                "rank": result.rank,
                "document_id": result.doc_id,
                "categories": ", ".join(result.categories),
                "score": result.score,
                "preview": result.preview,
            }
            for result in results
        ]
    )

## 10. Dataset selection and document collection

The application uses six business-news categories from the Reuters Corpus. A Reuters article may belong to more than one category, so duplicate document IDs are removed while constructing the searchable document collection.

In [ ]:
documents = load_reuters_document_collection(DEFAULT_CATEGORIES)

print("Selected categories:", DEFAULT_CATEGORIES)
print("Number of unique documents:", len(documents))
print("Sample document ID:", documents[0]["id"])
print("Sample categories:", documents[0]["categories"])
print("Sample text preview:", documents[0]["text"][:300])

In [ ]:
category_df = summarise_document_categories(documents)
category_df

## 11. Build and inspect the preprocessing pipeline

The same preprocessing pipeline is applied to both stored documents and user queries:

1. Tokenise text with `RegexpTokenizer`
2. Convert tokens to lowercase
3. Remove English stop words
4. Apply stemming

Porter stemming is used in the main application. Snowball stemming is retained as a possible design alternative for later experimental comparison.

In [ ]:
sample_text = "The companies announced several acquisitions and were trading internationally."

porter = PorterStemmer()
snowball = SnowballStemmer("english")

sample_words = [
    "companies",
    "announced",
    "acquisitions",
    "trading",
    "internationally",
]

stemming_comparison = pd.DataFrame({
    "word": sample_words,
    "Porter": [porter.stem(word) for word in sample_words],
    "Snowball": [snowball.stem(word) for word in sample_words],
})

stemming_comparison

In [ ]:
# Build the complete Project 1 information retrieval application.
ir_application = InformationRetrievalApplication(
    documents=documents,
    stemmer_name="porter",
    k1=1.5,
    b=0.75,
)

print("Original text:")
print(sample_text)
print("\nPreprocessed tokens:")
print(ir_application.preprocessor.preprocess_text(sample_text))
print("\nVocabulary size in inverted index:", len(ir_application.inverted_index))
print("TF-IDF matrix shape:", ir_application.tfidf_matrix.shape)

## 12. Dataset and term-frequency analysis

The application analyses frequent processed terms in the selected document collection. This provides evidence for the report's dataset-description section and helps identify high-frequency terms that may have limited retrieval value.

In [ ]:
frequency_df = pd.DataFrame(
    ir_application.analyse_term_frequency(top_n=20),
    columns=["term", "frequency"],
)
frequency_df

In [ ]:
ax = frequency_df.plot(
    x="term",
    y="frequency",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)
ax.set_title("Top 20 Terms after Preprocessing")
ax.set_xlabel("Term")
ax.set_ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 13. Document indexing

An **inverted index** maps each processed term to the documents containing that term. During query processing, the index identifies candidate documents before a ranking score is calculated. This avoids scoring documents that contain none of the processed query terms.

In [ ]:
sample_terms = ["compani", "oil", "trade"]

for term in sample_terms:
    matching_documents = ir_application.inverted_index.get(term, [])
    print(f"{term!r}: {len(matching_documents)} matching documents")
    print("First five document indices:", matching_documents[:5])

## 14. Retrieval method 1: TF-IDF and cosine similarity

TF-IDF represents documents and queries in a shared vector space. Cosine similarity measures the relationship between the query vector and each candidate-document vector, producing a ranked result list.

In [ ]:
test_query = "company acquisition merger"

tfidf_results = ir_application.rank_documents_tfidf(test_query, top_k=5)
format_retrieval_results(tfidf_results)

## 15. Retrieval method 2: BM25

BM25 is implemented as an alternative relevance-ranking method. It includes:

- term-frequency saturation, so repeated terms provide diminishing returns; and
- document-length normalisation, so long documents are not rewarded solely for containing more words.

The application uses `k1=1.5` and `b=0.75`. These parameters should be identified as configurable design decisions and discussed as a limitation if they are not tuned experimentally.

In [ ]:
bm25_results = ir_application.rank_documents_bm25(test_query, top_k=5)
format_retrieval_results(bm25_results)

## 16. Query processing and WordNet expansion

The query-processing module applies POS tagging and restricts WordNet expansion to nouns. The number of added synonyms is capped to reduce query drift. The expanded query is then processed through the same text pipeline and ranked with BM25.

In [ ]:
expanded_terms, expanded_tokens = ir_application.expand_query_with_wordnet(
    test_query,
    max_synonyms_per_noun=2,
)

print("Original query:", test_query)
print("Expanded raw terms:", expanded_terms)
print("Processed expanded tokens:", expanded_tokens)

expanded_results = ir_application.rank_documents_bm25_with_query_expansion(
    test_query,
    top_k=5,
)
format_retrieval_results(expanded_results)

## 17. Retrieved-document entity extraction

Named-entity recognition provides additional information about retrieved articles. The application can display selected entity types:

- `PERSON`
- `ORG`
- `GPE`
- `DATE`

Entity extraction is a result-presentation feature and does not alter the retrieval score.

In [ ]:
nlp = load_spacy_model()

if nlp is None:
    print("spaCy model en_core_web_sm is not installed.")
    print("Run: python -m spacy download en_core_web_sm")
else:
    entity_sample = """
    The Lions will host Melbourne Victory at Kingston on Sunday afternoon
    in their Hahn Australia Cup clash.
    """
    print(extract_document_entities(entity_sample, nlp))

In [ ]:
if nlp is not None and bm25_results:
    top_document = documents[bm25_results[0].doc_index]
    entities = extract_document_entities(top_document["text"], nlp)

    print("Top result:", top_document["id"])
    print("Named entities:")
    for entity_text, entity_label in entities[:15]:
        print(f"{entity_text:<40} {entity_label}")

## 18. Retrieved-document sentiment analysis

VADER sentiment analysis is included as an optional attribute for retrieved articles. It is not used to determine relevance because sentiment polarity and topical relevance are different concepts. Keeping them separate prevents a positive or negative article from receiving a higher retrieval score solely because of its sentiment.

In [ ]:
sentiment_analyser = initialise_sentiment_analyser()

if sentiment_analyser is None:
    print("VADER lexicon is missing. Run: nltk.download('vader_lexicon')")
elif bm25_results:
    top_document = documents[bm25_results[0].doc_index]
    label, scores = analyse_document_sentiment(
        top_document["text"],
        sentiment_analyser,
    )
    print("Document:", top_document["id"])
    print("Sentiment label:", label)
    print("VADER scores:", scores)

## 19. Performance evaluation

Reuters category labels are used as reproducible proxy relevance judgements. For example, documents tagged `acq` are treated as relevant to the acquisition query.

This proxy is approximate: a category label does not guarantee relevance to every query about that category, and a relevant article may use a different label. This limitation must be discussed in the report.

The implemented evaluation metrics are:

- Precision@10
- Recall@10
- Average Precision@10
- MAP@10

In [ ]:
TEST_QUERIES = {
    "acq": "company acquisition merger",
    "crude": "crude oil petroleum prices",
    "trade": "trade deficit tariffs",
    "money-fx": "currency exchange rate",
    "interest": "interest rate central bank",
    "ship": "shipping vessel cargo",
}

K = 10

evaluation_details, evaluation_summary = evaluate_retrieval_performance(
    application=ir_application,
    test_queries=TEST_QUERIES,
    k=K,
)

evaluation_details

In [ ]:
evaluation_summary

In [ ]:
details_path = RESULTS_DIR / "evaluation_details.csv"
summary_path = RESULTS_DIR / "evaluation_summary.csv"

evaluation_details.to_csv(details_path, index=False)
evaluation_summary.to_csv(summary_path, index=False)

print("Saved:", details_path)
print("Saved:", summary_path)

In [ ]:
metric_columns = ["precision@10", "recall@10", "MAP@10"]
plot_df = evaluation_summary.set_index("method")[metric_columns]

ax = plot_df.plot(kind="bar", figsize=(10, 6))
ax.set_title("Retrieval Method Comparison")
ax.set_xlabel("Method")
ax.set_ylabel("Mean score")
ax.set_ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()

figure_path = RESULTS_DIR / "retrieval_method_comparison.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", figure_path)

## 20. Select the default ranking method

The following cell identifies the method with the highest observed MAP@10. The final report must still interpret the detailed results, explain possible causes, and avoid relying only on the overall average.

In [ ]:
best_row = evaluation_summary.sort_values("MAP@10", ascending=False).iloc[0]

print(
    f"Highest observed MAP@10: {best_row['method']} "
    f"({best_row['MAP@10']:.3f})."
)
print(
    "Review evaluation_details before deciding whether this method should be "
    "recommended as the final default."
)

## 21. User-facing information retrieval application

The application supports all three evaluated ranking methods. Named entities and sentiment are optional display attributes for each retrieved document.

In [ ]:
def display_ranked_documents(
    query_text,
    method="bm25",
    top_k=10,
    show_entities=True,
    show_sentiment=True,
):
    method = method.lower().strip()

    if method == "tfidf":
        results = ir_application.rank_documents_tfidf(query_text, top_k=top_k)
    elif method == "bm25":
        results = ir_application.rank_documents_bm25(query_text, top_k=top_k)
    elif method in {"expanded", "bm25_expanded"}:
        results = ir_application.rank_documents_bm25_with_query_expansion(
            query_text,
            top_k=top_k,
        )
    else:
        raise ValueError("method must be: tfidf, bm25, or expanded")

    if not results:
        print("No matching documents were found.")
        return

    print(f"Query: {query_text}")
    print(f"Ranking method: {method}")
    print("=" * 90)

    for result in results:
        document = documents[result.doc_index]
        print(
            f"{result.rank}. score={result.score:.4f} | "
            f"id={result.doc_id} | categories={list(result.categories)}"
        )
        print("   Preview:", result.preview + "...")

        if show_entities and nlp is not None:
            entities = extract_document_entities(document["text"], nlp)[:8]
            if entities:
                print("   Entities:", entities)

        if show_sentiment and sentiment_analyser is not None:
            label, scores = analyse_document_sentiment(
                document["text"],
                sentiment_analyser,
            )
            print(
                f"   Sentiment: {label} "
                f"(compound={scores['compound']:.3f})"
            )

        print()


display_ranked_documents(
    query_text="oil price and shipping company",
    method="bm25",
    top_k=5,
)

In [ ]:
def run_information_retrieval_application():
    print("=== KIT719 Project 1 Information Retrieval Application ===")
    print("Ranking methods: tfidf, bm25, expanded")
    print("Type 'quit' to exit.")

    while True:
        query_text = input("\nEnter query: ").strip()
        if query_text.lower() == "quit":
            print("Application closed.")
            break
        if not query_text:
            print("Please enter a non-empty query.")
            continue

        method = input("Choose ranking method [bm25]: ").strip().lower() or "bm25"

        try:
            display_ranked_documents(
                query_text=query_text,
                method=method,
                top_k=10,
            )
        except ValueError as error:
            print(error)


# Run manually because input() blocks 'Run All':
# run_information_retrieval_application()

## 22. Required report evidence and discussion

After running all experiments, the group should use the generated outputs to address:

1. Why Reuters and the six categories were selected.
2. Why the selected tokenisation, stop-word removal and stemming methods were used.
3. How the inverted index supports efficient candidate retrieval.
4. How TF-IDF/cosine similarity and BM25 differ.
5. Whether WordNet expansion improved retrieval or introduced query drift.
6. Why Reuters categories are only approximate relevance judgements.
7. Which ranking method performed best for each query and overall.
8. Why entity and sentiment analysis were kept separate from relevance scoring.
9. Limitations, including the small test-query set, proxy relevance labels, untuned BM25 parameters and general-purpose WordNet senses.
10. Future improvements, such as more test queries, manual relevance judgements, parameter tuning, spelling correction and domain-specific query expansion.

### Academic-integrity reminder
Do not fabricate metric values or conclusions. Run the notebook, preserve the generated evidence, understand each function, and record all AI assistance in the report appendix.